# M11 lab solution: Document Ingestion & Metadata Design

**Brief.** Multi-source ingestor: load a mixed folder (PDF, DOCX, Markdown, CSV, one web page) into Documents that share one metadata schema (source, type, section, page, date).

Run with the default environment: `uv run jupyter lab`. No model calls: loading and metadata only.

Open this notebook from `lab/solution/`, so `../data/` resolves to the lab's own data folder.

## Setup

In [ ]:
import re
import zipfile
from collections import Counter
from pathlib import Path

from bs4 import BeautifulSoup
from langchain_community.document_loaders import BSHTMLLoader, CSVLoader, Docx2txtLoader, PyPDFLoader, TextLoader
from langchain_core.documents import Document
from langchain_text_splitters import MarkdownHeaderTextSplitter

MIXED = Path("../data/mixed")
SCHEMA = ("source", "type", "section", "page", "date")
for p in sorted(MIXED.iterdir()):
    print(f"{p.name:35} {p.stat().st_size:>8} bytes")

## 1. One schema for every loader

Every loader names its metadata differently (PyPDF counts `page` from 0, CSV rows carry `row`, the HTML loader keeps `title`). `to_schema` maps each one onto the five fields at load time, so later stages can filter and cite without knowing which loader produced a chunk. `page` is 1-based where the source has pages and `None` where it has none; `date` is an ISO date taken from the file itself.

In [ ]:
def to_schema(doc, *, source, type, section, page, date):
    return Document(page_content=doc.page_content,
                    metadata={"source": source, "type": type, "section": section, "page": page, "date": date})


def first_heading(text):
    """First short line that is not a URL: a stand-in section title for a PDF page."""
    for line in text.splitlines():
        line = line.strip()
        if 3 < len(line) < 80 and not line.startswith("http"):
            return line
    return None


def load_pdf(path):
    pages = PyPDFLoader(str(path)).load()
    return [to_schema(p, source=path.name, type="pdf", section=first_heading(p.page_content),
                      page=p.metadata["page"] + 1, date=p.metadata.get("creationdate", "")[:10] or None)
            for p in pages]


def load_docx(path):
    (doc,) = Docx2txtLoader(str(path)).load()
    core = zipfile.ZipFile(path).read("docProps/core.xml").decode()
    created = re.search(r"<dcterms:created[^>]*>([^<]+)<", core)
    title = next(line for line in doc.page_content.splitlines() if line.strip())
    return [to_schema(doc, source=path.name, type="docx", section=title.strip(), page=None,
                      date=created.group(1)[:10] if created else None)]


def load_markdown(path):
    (doc,) = TextLoader(str(path)).load()
    _, front, body = doc.page_content.split("---", 2)
    meta = dict(line.split(": ", 1) for line in front.strip().splitlines())
    sections = MarkdownHeaderTextSplitter([("#", "h1"), ("##", "h2")]).split_text(body)
    return [to_schema(s, source=path.name, type="markdown", section=s.metadata.get("h2") or s.metadata.get("h1"),
                      page=None, date=meta.get("date")) for s in sections]


def load_csv(path):
    rows = CSVLoader(str(path), metadata_columns=["ticket_id", "date", "category"]).load()
    return [to_schema(r, source=path.name, type="csv", section=r.metadata["category"], page=None, date=r.metadata["date"])
            for r in rows]


def load_html(path):
    (doc,) = BSHTMLLoader(str(path)).load()
    meta = BeautifulSoup(path.read_text(), "html.parser").find("meta", attrs={"name": "date"})
    return [to_schema(doc, source=path.name, type="html", section=doc.metadata.get("title"), page=None,
                      date=meta["content"] if meta else None)]


LOADERS = {".pdf": load_pdf, ".docx": load_docx, ".md": load_markdown, ".csv": load_csv, ".html": load_html}

## 2. Load the folder

In [ ]:
docs = [d for p in sorted(MIXED.iterdir()) if p.suffix in LOADERS for d in LOADERS[p.suffix](p)]
print(len(docs), "documents")
print(Counter(d.metadata["type"] for d in docs))

In [ ]:
shown = set()
for d in docs:
    if d.metadata["type"] not in shown:
        shown.add(d.metadata["type"])
        print(d.metadata)
        print("   ", d.page_content[:120].replace("\n", " "), "\n")

## 3. Schema check

`check_schema` fails when any document lacks one of the five keys or has an empty `source` or `type`. It passes on the loaded folder; the second cell shows it failing on a document that a careless loader built without a `date`.

In [ ]:
def check_schema(docs):
    problems = []
    for i, d in enumerate(docs):
        missing = [k for k in SCHEMA if k not in d.metadata]
        empty = [k for k in ("source", "type") if not d.metadata.get(k)]
        if missing or empty:
            problems.append(f"doc {i} ({d.metadata.get('source')}): missing {missing}, empty {empty}")
    assert not problems, "schema check failed:\n" + "\n".join(problems)
    return len(docs)


print("schema check passed on", check_schema(docs), "documents")

In [ ]:
broken = Document(page_content="orphan", metadata={"source": "manual.txt", "type": "text", "section": None, "page": None})
try:
    check_schema(docs + [broken])
except AssertionError as err:
    print(err)

## Results

In [ ]:
import pandas as pd

table = pd.DataFrame([d.metadata for d in docs])
print(table.groupby("type").agg(documents=("source", "size"), files=("source", "nunique"),
                                with_section=("section", "count"), with_page=("page", "count"), with_date=("date", "count")))

## What this shows

- Five loaders, five metadata dialects, one schema. The mapping lives at load time; adding a field later would mean re-ingesting every file.
- Element-level loading changes what a Document is: the Markdown guide becomes one Document per section, the CSV one per ticket, the PDF one per page, while the DOCX and HTML pages stay whole. The counts per type above come from those choices.
- `page` and `section` are honest about what each format has: `None` where a format has no pages, and a heuristic title for PDF pages, which is worth checking by eye in the printed records.